<a href="https://colab.research.google.com/github/amvicioushecs/FinesseStories-CLI/blob/main/FinesseStories_CLI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python3
"""
FinesseStories CLI (finessestories.py)
--------------------------------------------------
A dedicated CLI and REPL Shell that implements the 4-Stage "AI Fiction Book Writer"
architecture. It orchestrates high-continuity novel creation while enforcing the
loop-prevention and audit guardrails.

Pipeline Archetype:
- Stage 1: Context Intake & Knowledge Mining (knowledge-miner)
- Stage 2: Core Source of Truth (story-bible-architect, granular-outliner)
- Stage 3: Recursive Manuscript Generation (chapter-drafter, continuity-auditor)
- Stage 4: Verification, Polish, & Final Export (prose-polisher, originality-checker)

Author Control & Loop Prevention:
- Automatically tracks sliding summaries of preceding chapters.
- Runs an independent continuity auditor checking against locked world rules.
- Captures file integrity changes to prevent repetitive phrasing and loop thrashing.
- Connects to OpenRouter (defaulting to the highly efficient Free Tier Router).
"""

import os
import sys
import json
import argparse
import hashlib
import time
import re
import urllib.request
from typing import Dict, List, Any, Tuple, Optional

CLR_CYAN = "\033[96m"
CLR_GREEN = "\033[92m"
CLR_YELLOW = "\033[93m"
CLR_RED = "\033[91m"
CLR_MAGENTA = "\033[95m"
CLR_BOLD = "\033[1m"
CLR_RESET = "\033[0m"

PROJECT_FILE = ".finessestories_project.json"
BIBLE_FILE = "finessestories_bible.json"
DRAFTS_DIR = "drafts"

DEFAULT_PROJECT_SCHEMA = {
    "project_metadata": {
        "title": "Untitled Masterpiece",
        "genre": "Unspecified",
        "tone_and_style": "Standard fiction",
        "current_chapter": 1,
        "token_budget_usd": 5.00,
        "accumulated_cost_usd": 0.00
    },
    "project_brief": {
        "core_premise": "",
        "primary_themes": [],
        "target_tone": "",
        "primary_entities": [],
        "user_mandates": [],
        "world_dossier": "",
        "character_roster": ""
    },
    "chapters": {}  # Maps "chapter_1": {"title": "", "pov": "", "outline": {}, "draft": "", "polished": "", "summary": ""}
}

def log_status(color: str, prefix: str, message: str) -> None:
    """Helper to print beautifully structured colored logs for the terminal."""
    print(f"{CLR_BOLD}{color}[{prefix}]{CLR_RESET} {message}")

def load_project_state() -> Dict[str, Any]:
    """Loads the active story session project metadata file or returns a default schema."""
    if not os.path.exists(PROJECT_FILE):
        return dict(DEFAULT_PROJECT_SCHEMA)
    try:
        with open(PROJECT_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    except json.JSONDecodeError as e:
        log_status(CLR_RED, "JSON_ERR", f"Failed parsing Project State: {e}")
        sys.exit(1)

def save_project_state(data: Dict[str, Any]) -> None:
    """Saves the project data back to its master JSON session state."""
    with open(PROJECT_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

def load_story_bible() -> Dict[str, Any]:
    """Loads the current Story Bible (SSOT). Returns empty schema if not found."""
    if not os.path.exists(BIBLE_FILE):
        return {
            "story_bible_version": "1.0",
            "project_title": "Untitled Project",
            "canon_rules": [],
            "world": {
                "magic_tech_system": {"name": "", "rules": [], "costs": [], "limitations": []},
                "locations": [],
                "factions": []
            },
            "characters": [],
            "chronology_and_canon_ledger": []
        }
    try:
        with open(BIBLE_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    except json.JSONDecodeError as e:
        log_status(CLR_RED, "JSON_ERR", f"Failed parsing Story Bible file: {e}")
        return {}

def save_story_bible(data: Dict[str, Any]) -> None:
    """Saves the master Story Bible file back to disk."""
    with open(BIBLE_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

def request_openrouter_api(api_key: str, system_prompt: str, user_prompt: str, model: str = "openrouter/free") -> str:
    """
    Submits standard chat requests to the OpenRouter gateway.
    Handles fallbacks gracefully, notifying users of network/key anomalies.
    """
    if not api_key:
        log_status(CLR_YELLOW, "OFFLINE_MODE", "No OpenRouter API Key was found. Simulating model generation outputs...")
        return "[MOCKED RESPONSE: Please run CLI commands with a valid OpenRouter API Key using the --key option or configure a session shell.]"

    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://github.com/finessestories/cli",
        "X-Title": "FinesseStories Narrative CLI"
    }

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    }

    req = urllib.request.Request(url, data=json.dumps(payload).encode("utf-8"), headers=headers, method="POST")
    try:
        with urllib.request.urlopen(req) as response:
            res_body = json.loads(response.read().decode("utf-8"))
            choices = res_body.get("choices", [])
            if choices:
                return choices[0].get("message", {}).get("content", "").strip()
            return f"Error: No content generated. Raw response: {res_body}"
    except urllib.error.HTTPError as e:
        err_msg = e.read().decode("utf-8")
        log_status(CLR_RED, "API_ERR", f"HTTP {e.code} Error: {err_msg}")
        return f"Error: API status {e.code}"
    except Exception as e:
        log_status(CLR_RED, "CONN_ERR", f"Failed to connect to OpenRouter: {e}")
        return "Error: Connection exception encountered."

def run_stage1_ingest(api_key: str, reference_text_path: str) -> None:
    """
    Stage 1: knowledge-miner
    Reads reference notes, outlines, or transcripts, and extracts the foundational Brief.
    """
    log_status(CLR_CYAN, "STAGE_1", f"Loading and mining intake source: {reference_text_path}")
    if not os.path.exists(reference_text_path):
        log_status(CLR_RED, "FILE_NOT_FOUND", f"The reference file path '{reference_text_path}' does not exist.")
        return

    try:
        with open(reference_text_path, "r", encoding="utf-8") as f:
            raw_content = f.read()
    except IOError as e:
        log_status(CLR_RED, "READ_ERR", f"Failed reading raw source material: {e}")
        return

    system_prompt = (
        "You are an expert literary development editor and narrative analyst.\n"
        "Task: Analyze the attached context materials and extract the core seed elements for a novel.\n"
        "Extract and structure the following into a clean, markdown-friendly Project Brief."
    )
    user_prompt = (
        "Core Task: Parse this material into raw premise, themes, tone preferences, and primary entities.\n"
        "Ensure you explicitly flag any structural contradictions with a [CONTRADICTION] tag.\n\n"
        f"=== INGESTED SOURCE ===\n{raw_content[:8000]}\n=====================\n\n"
        "Output standard format:\n"
        "Core Premise: [1-2 sentences]\n"
        "Primary Themes & Tropes: [Bullet list]\n"
        "Target Tone & Style: [Brief description]\n"
        "Primary Entities: [List characters, locations, systems]\n"
        "User Mandates: [Identify non-negotiable details]"
    )

    log_status(CLR_CYAN, "API_CALL", "Transmitting source payload to knowledge-miner subagent...")
    brief_markdown = request_openrouter_api(api_key, system_prompt, user_prompt)

    # Save the output to state
    project = load_project_state()
    project["project_brief"]["core_premise"] = brief_markdown
    save_project_state(project)

    # Output brief to console
    print(f"\n{CLR_BOLD}{CLR_GREEN}=== PARSED PROJECT BRIEF ==={CLR_RESET}\n")
    print(brief_markdown)
    print(f"\n{CLR_GREEN}Project Brief successfully committed to Session State File (.{PROJECT_FILE}){CLR_RESET}\n")

def run_stage2_world(api_key: str) -> None:
    """
    Stage 2a: world-building-architect
    Constructs a watertight, rule-governed setting and systems dossier.
    """
    project = load_project_state()
    brief = project["project_brief"]["core_premise"]

    if not brief:
        log_status(CLR_RED, "MISSING_BRIEF", "No brief found in session state. Run stage 1 (/ingest) first.")
        return

    log_status(CLR_CYAN, "STAGE_2_WORLD", "Running World-Building Architect to engineer settings and systems...")

    system_prompt = (
        "You are a premier fantasy/sci-fi world-builder and narrative setting architect.\n"
        "Task: Take the provided Project Brief and engineer a complete, immersive Setting & Systems Dossier."
    )
    user_prompt = (
        "Generate a complete, deeply detailed world setting dossier covering the following 5 dimensions:\n"
        "1. Physical & Spatial Architecture (Primary settings with visual layouts, smells, and sounds)\n"
        "2. Magic / Tech / System Rules & Hard Constraints (Mechanics, hard limits, costs, failure modes)\n"
        "3. Factions, Power Dynamics, & Economy (Guilds/corporations, frictions, societal taboos)\n"
        "4. Deep Lore & Historical Anchors (Catalyst events, public knowledge vs. hidden truths)\n"
        "5. World Constraints (5 explicit 'Rules of the World' that cannot be violated)\n\n"
        f"=== PROJECT BRIEF ===\n{brief}\n=====================\n\n"
        "Output clean, evocative Markdown. Avoid generic tropes."
    )

    log_status(CLR_CYAN, "API_CALL", "Transmitting brief to World-Building Architect...")
    dossier = request_openrouter_api(api_key, system_prompt, user_prompt)

    project["project_brief"]["world_dossier"] = dossier
    save_project_state(project)

    print(f"\n{CLR_BOLD}{CLR_GREEN}=== GENERATED WORLD DOSSIER ==={CLR_RESET}\n")
    print(dossier)
    log_status(CLR_GREEN, "SUCCESS", "Setting and systems dossier saved to project session state.")

def run_stage2_characters(api_key: str) -> None:
    """
    Stage 2b: character-archetype-builder
    Generates complex psychological Cast profiles, Wants/Needs/Lies, and unique voice profiles.
    """
    project = load_project_state()
    brief = project["project_brief"]["core_premise"]
    world_dossier = project["project_brief"].get("world_dossier", "")

    if not brief:
        log_status(CLR_RED, "MISSING_BRIEF", "No brief found in session state. Run stage 1 (/ingest) first.")
        return
    if not world_dossier:
        log_status(CLR_YELLOW, "WARNING", "No World Dossier found. It is highly recommended to run /world first to ground character flaws.")

    log_status(CLR_CYAN, "STAGE_2_CHARACTERS", "Running Psychological Character Archetype Builder...")

    system_prompt = (
        "You are a lead character developer and psychological profiler for bestselling fiction.\n"
        "Task: Build a comprehensive Character Master Roster for the primary cast (Protagonist, Antagonist, Key Support)."
    )
    user_prompt = (
        "Create a comprehensive Character Master Roster detailing:\n"
        "1. Identity & External Profile (Physical blueprints, behavioral quirks under stress)\n"
        "2. Psychological Engine (The Want/External Goal, the Lie/Internal Flaw, the Need/Internal Growth, the Wound/Backstory Catalyst)\n"
        "3. Dialogue & Linguistic Profile (Vocabulary level, sentence rhythm, unique cadences, speech taboos, and reaction benchmarks)\n"
        "4. Relationship Tension Matrix (Dynamics, hidden resentments, or loyalty bounds)\n\n"
        f"=== PROJECT BRIEF ===\n{brief}\n\n"
        f"=== WORLD DOSSIER ===\n{world_dossier}\n=====================\n\n"
        "Output clean, highly structural Markdown. Ensure dialogue and quirks distinguish each character cleanly."
    )

    log_status(CLR_CYAN, "API_CALL", "Transmitting dossiers to Character Profiler subagent...")
    roster = request_openrouter_api(api_key, system_prompt, user_prompt)

    project["project_brief"]["character_roster"] = roster
    save_project_state(project)

    print(f"\n{CLR_BOLD}{CLR_GREEN}=== GENERATED CHARACTER ROSTER ==={CLR_RESET}\n")
    print(roster)
    log_status(CLR_GREEN, "SUCCESS", "Character Master Roster saved to project session state.")

def run_stage2_architect(api_key: str) -> None:
    """
    Stage 2c: story-bible-compiler (Master Canon Integrator)
    Merges the generated Setting Dossier and Character Roster into a unified SSOT JSON.
    """
    project = load_project_state()
    world_dossier = project["project_brief"].get("world_dossier", "")
    character_roster = project["project_brief"].get("character_roster", "")

    if not world_dossier or not character_roster:
        log_status(CLR_YELLOW, "MISSING_DOSSIERS", "Missing detailed dossiers. Using Project Brief as a fallback.")
        # Fallback to building directly from the raw brief if dossiers aren't generated
        brief = project["project_brief"]["core_premise"]
        if not brief:
            log_status(CLR_RED, "MISSING_INPUT", "No source context available. Run /ingest, /world, and /characters first.")
            return
        world_context = f"Project Brief: {brief}"
        char_context = f"Project Brief: {brief}"
    else:
        world_context = world_dossier
        char_context = character_roster

    log_status(CLR_CYAN, "STAGE_2C", "Running Master Canon Integrator to compile dossiers into JSON SSOT...")

    system_prompt = (
        "You are a data architect specializing in knowledge systems for generative narrative engines.\n"
        "Task: Merge the provided World Dossier and Character Master Roster into a single, clean, structured JSON Master Story Bible."
    )
    user_prompt = (
        "Compile the provided World and Character dossiers into a single valid JSON matching this exact structure:\n"
        "{\n"
        "  \"story_bible_version\": \"1.0\",\n"
        "  \"project_title\": \"[Insert Title]\",\n"
        "  \"canon_rules\": [\"Rule 1\", \"Rule 2\"],\n"
        "  \"world\": {\n"
        "    \"magic_tech_system\": {\"name\": \"\", \"rules\": [], \"costs\": [], \"limitations\": []},\n"
        "    \"locations\": [{\"id\": \"loc_01\", \"name\": \"\", \"sensory_anchors\": [], \"description\": \"\"}],\n"
        "    \"factions\": [{\"name\": \"\", \"conflict\": \"\"}]\n"
        "  },\n"
        "  \"characters\": [{\"id\": \"char_01\", \"name\": \"\", \"role\": \"\", \"psychology\": {\"want\": \"\", \"need\": \"\", \"lie\": \"\"}, \"voice_profile\": {\"cadence\": \"\", \"vocabulary\": \"\", \"taboos\": []}}]\n"
        "}\n\n"
        f"=== WORLD DOSSIER ===\n{world_context}\n\n"
        f"=== CHARACTER ROSTER ===\n{char_context}\n========================\n\n"
        "Output ONLY valid JSON code. No markdown wrap, no explanations, no text preceding or following the JSON block."
    )

    log_status(CLR_CYAN, "API_CALL", "Transmitting dossiers to Canon Integrator subagent...")
    raw_json_str = request_openrouter_api(api_key, system_prompt, user_prompt)

    cleaned_json = raw_json_str.strip()
    if cleaned_json.startswith("```json"):
        cleaned_json = cleaned_json.replace("```json", "", 1)
    if cleaned_json.endswith("```"):
        cleaned_json = cleaned_json.rsplit("```", 1)[0]
    cleaned_json = cleaned_json.strip()

    try:
        bible_data = json.loads(cleaned_json)
        save_story_bible(bible_data)
        log_status(CLR_GREEN, "SUCCESS", f"Master Story Bible compiled and saved to '{BIBLE_FILE}'")

        project["project_metadata"]["title"] = bible_data.get("project_title", "Untitled Story")
        save_project_state(project)
    except json.JSONDecodeError:
        log_status(CLR_RED, "PARSE_ERR", "Model failed to return valid JSON syntax. Saving raw text as fallback...")
        fallback_data = {
            "story_bible_version": "1.0",
            "project_title": "Parsing Fallback Project",
            "raw_architect_dump": raw_json_str
        }
        save_story_bible(fallback_data)

def run_stage2_outliner(api_key: str) -> None:
    """
    Stage 2: granular-outliner
    Takes the compiled Story Bible and constructs a multi-act detailed outline.
    """
    bible = load_story_bible()
    if not bible or "story_bible_version" not in bible:
        log_status(CLR_RED, "MISSING_BIBLE", "Story Bible is missing or invalid. Initialize Stage 2a first.")
        return

    log_status(CLR_CYAN, "STAGE_2B", "Generating 3-Act granular chapter outline (granular-outliner)...")

    system_prompt = (
        "You are a structural story editor specializing in perfectly paced narrative arcs.\n"
        "Task: Generate a detailed Chapter Outline Matrix (Chapters 1-3) based on the Story Bible."
    )
    user_prompt = (
        "Analyze the world and characters in the Story Bible below and create an outline structured EXACTLY like this for Chapters 1-3:\n"
        "{\n"
        "  \"chapter_1\": {\n"
        "    \"title\": \"Title of Chapter 1\",\n"
        "    \"pov_character\": \"Elena Voss\",\n"
        "    \"narrative_goal\": \"What changes from start to finish?\",\n"
        "    \"scene_sequence\": [\"Scene A: ...\", \"Scene B: ...\", \"Scene C: ...\"],\n"
        "    \"canon_checklist\": [\"Rule or character arc advanced\"],\n"
        "    \"pacing_tension_index\": 4\n"
        "  },\n"
        "  \"chapter_2\": { ... },\n"
        "  \"chapter_3\": { ... }\n"
        "}\n\n"
        f"=== MASTER STORY BIBLE ===\n{json.dumps(bible, indent=2)}\n==========================\n"
        "Output ONLY valid JSON. No conversational text."
    )

    log_status(CLR_CYAN, "API_CALL", "Transmitting Story Bible to Outliner subagent...")
    raw_outline_str = request_openrouter_api(api_key, system_prompt, user_prompt)

    cleaned_outline = raw_outline_str.strip()
    if cleaned_outline.startswith("```json"):
        cleaned_outline = cleaned_outline.replace("```json", "", 1)
    if cleaned_outline.endswith("```"):
        cleaned_outline = cleaned_outline.rsplit("```", 1)[0]
    cleaned_outline = cleaned_outline.strip()

    try:
        outline_data = json.loads(cleaned_outline)
        project = load_project_state()
        project["chapters"] = outline_data
        save_project_state(project)
        log_status(CLR_GREEN, "SUCCESS", "Granular Chapter Outline successfully loaded and synchronized with project state.")
    except json.JSONDecodeError:
        log_status(CLR_RED, "PARSE_ERR", "Could not parse Outliner response as clean JSON. Inspect raw response.")
        print(raw_outline_str)

def run_stage3_draft(api_key: str, chapter_num: int) -> None:
    """
    Stage 3: chapter-drafter
    Drafts rich, immersive prose for a target chapter while keeping track of sliding context.
    """
    project = load_project_state()
    bible = load_story_bible()

    ch_key = f"chapter_{chapter_num}"
    if ch_key not in project["chapters"]:
        log_status(CLR_RED, "MISSING_OUTLINE", f"No outline registered for Chapter {chapter_num}. Run Stage 2b first.")
        return

    log_status(CLR_CYAN, "STAGE_3A", f"Drafting raw manuscript prose for Chapter {chapter_num} (chapter-drafter)...")

    # Get Previous Chapter Summary (Sliding Context Window rule)
    prev_chapter_summary = "No preceding chapters yet. This is the opening chapter of the book."
    if chapter_num > 1:
        prev_ch_key = f"chapter_{chapter_num - 1}"
        if prev_ch_key in project["chapters"]:
            prev_chapter_summary = project["chapters"][prev_ch_key].get("summary", "Summary of events from Chapter 1 is unknown.")

    chapter_outline = project["chapters"][ch_key]

    system_prompt = (
        "You are a master fiction author writing in the established tone and style of this project.\n"
        "Task: Write a full, immersive narrative chapter. Show, don't tell. Establish deep sensory anchors."
    )
    user_prompt = (
        "Write the full prose draft for the targeted chapter based on the provided inputs:\n\n"
        f"[STORY BIBLE]:\n{json.dumps(bible, indent=2)}\n\n"
        f"[TARGET CHAPTER OUTLINE]:\n{json.dumps(chapter_outline, indent=2)}\n\n"
        f"[PREVIOUS CHAPTER SUMMARY]:\n{prev_chapter_summary}\n\n"
        "Rules to follow during drafting:\n"
        "1. Strictly match character voice metrics, accent tokens, and dialogue taboos defined in the Story Bible.\n"
        "2. Fulfill all required Scene A, Scene B, and Scene C milestones from the chapter's outline.\n"
        "3. Incorporate at least 3 active sensory anchors (smells, sounds, textures) directly in your descriptions.\n"
        "4. Do NOT output commentary or introductory remarks. Output ONLY the story prose for this chapter."
    )

    log_status(CLR_CYAN, "API_CALL", "Transmitting chapter outlines and sliding context block...")
    drafted_prose = request_openrouter_api(api_key, system_prompt, user_prompt)

    # Save to Session State
    project["chapters"][ch_key]["draft"] = drafted_prose
    save_project_state(project)

    # Save to an external raw draft file
    os.makedirs(DRAFTS_DIR, exist_ok=True)
    draft_file_path = os.path.join(DRAFTS_DIR, f"chapter_{chapter_num}_draft.txt")
    with open(draft_file_path, "w", encoding="utf-8") as f:
        f.write(drafted_prose)

    log_status(CLR_GREEN, "SUCCESS", f"Drafted prose saved to project state and '{draft_file_path}'")

def run_stage3_auditor(api_key: str, chapter_num: int) -> None:
    """
    Stage 3: continuity-auditor
    Audits drafted chapter against locked world rules, and outputs suggestions and canon updates.
    """
    project = load_project_state()
    bible = load_story_bible()

    ch_key = f"chapter_{chapter_num}"
    if ch_key not in project["chapters"] or not project["chapters"][ch_key].get("draft"):
        log_status(CLR_RED, "MISSING_DRAFT", f"No raw draft found for Chapter {chapter_num}. Run Stage 3a first.")
        return

    log_status(CLR_CYAN, "STAGE_3B", f"Running Continuity Audit for Chapter {chapter_num} (continuity-auditor)...")

    draft = project["chapters"][ch_key]["draft"]
    chapter_outline = project["chapters"][ch_key]

    system_prompt = (
        "You are a ruthless continuity editor and story bible guardian.\n"
        "Analyze the provided draft against the Story Bible and the chapter outline to check for rule violations."
    )
    user_prompt = (
        "Review the draft and identify character inconsistencies, lore breaches, or outline deviations.\n\n"
        f"=== STORY BIBLE CONSTRAINTS ===\n{json.dumps(bible, indent=2)}\n===============================\n\n"
        f"=== TARGET CHAPTER OUTLINE ===\n{json.dumps(chapter_outline, indent=2)}\n==============================\n\n"
        f"=== RAW CHAPTER DRAFT ===\n{draft[:8000]}\n=========================\n\n"
        "Your response MUST adhere strictly to the following syntax to facilitate parsing:\n"
        "STATUS: [PASS or FAIL]\n"
        "CONTINUITY_ERRORS: [List details of errors, or 'None']\n"
        "SUGGESTED_LINE_EDITS: [List suggested prose corrections]\n"
        "CANON_ADDITIONS: [List any new facts introduced in this chapter that need to be logged into the Bible]\n"
        "SUMMARY: [Provide a 150-word plot summary of this chapter for downstream context]"
    )

    log_status(CLR_CYAN, "API_CALL", "Transmitting draft text to Continuity Auditor subagent...")
    audit_report = request_openrouter_api(api_key, system_prompt, user_prompt)

    # Output to console
    print(f"\n{CLR_BOLD}{CLR_MAGENTA}=== CONTINUITY AUDIT REPORT (CHAPTER {chapter_num}) ==={CLR_RESET}\n")
    print(audit_report)

    # Parse and update the story bible's canon and chronology ledger dynamically
    canon_updates = []
    summary_text = ""
    for line in audit_report.splitlines():
        if line.startswith("CANON_ADDITIONS:"):
            updates_str = line.replace("CANON_ADDITIONS:", "").strip()
            if updates_str and updates_str.lower() != "none":
                canon_updates = [item.strip() for item in updates_str.split(";")]
        elif line.startswith("SUMMARY:"):
            summary_text = line.replace("SUMMARY:", "").strip()

    if canon_updates:
        log_status(CLR_CYAN, "BIBLE_UPDATE", "Updating Story Bible timeline ledger with newly audited canon...")
        for update in canon_updates:
            bible["chronology_and_canon_ledger"].append({
                "chapter": chapter_num,
                "canon_fact": update,
                "timestamp": int(time.time())
            })
        save_story_bible(bible)

    if summary_text:
        project["chapters"][ch_key]["summary"] = summary_text
        save_project_state(project)
        log_status(CLR_GREEN, "SUCCESS", "Sliding Chapter Summary updated successfully in session state.")

def run_stage4_polish(api_key: str, chapter_num: int) -> None:
    """
    Stage 4: prose-polisher
    Performs style polishing, removing passive verbs and cliché 'AI-isms'.
    """
    project = load_project_state()
    ch_key = f"chapter_{chapter_num}"
    if ch_key not in project["chapters"] or not project["chapters"][ch_key].get("draft"):
        log_status(CLR_RED, "MISSING_DRAFT", f"No raw draft found for Chapter {chapter_num}. Cannot run polish.")
        return

    log_status(CLR_CYAN, "STAGE_4A", f"Polishing and refining prose for Chapter {chapter_num} (prose-polisher)...")

    draft = project["chapters"][ch_key]["draft"]

    system_prompt = (
        "You are a premier book editor. Elevate raw draft prose into publishing-ready text by eliminating passive voice,\n"
        "monotonous cadence, filter words (noticed, saw, felt), and cliché AI-isms (e.g., 'tapestry', 'testament to', 'beacon of hope')."
    )
    user_prompt = (
        "Perform deep line editing and stylistic polishing on the provided prose. Keep the structural story beats exact,\n"
        "but maximize impact and emotional depth.\n\n"
        f"=== RAW DRAFT ===\n{draft}\n=================\n"
        "Output ONLY the polished final story text. Do not write introductory remarks or post-completion chat."
    )

    log_status(CLR_CYAN, "API_CALL", "Transmitting prose to editor engine...")
    polished_prose = request_openrouter_api(api_key, system_prompt, user_prompt)

    # Save Polished Draft
    project["chapters"][ch_key]["polished"] = polished_prose
    save_project_state(project)

    polished_file_path = os.path.join(DRAFTS_DIR, f"chapter_{chapter_num}_polished.txt")
    with open(polished_file_path, "w", encoding="utf-8") as f:
        f.write(polished_prose)

    log_status(CLR_GREEN, "SUCCESS", f"Polished manuscript saved in project session and '{polished_file_path}'")

def run_stage4_originality(api_key: str, chapter_num: int, reference_text_path: str) -> None:
    """
    Stage 4: originality-and-plagiarism-checker
    Scans generated prose against original reference materials to protect IP.
    """
    project = load_project_state()
    ch_key = f"chapter_{chapter_num}"

    polished_text = project["chapters"].get(ch_key, {}).get("polished", "")
    if not polished_text:
        polished_text = project["chapters"].get(ch_key, {}).get("draft", "")

    if not polished_text:
        log_status(CLR_RED, "MISSING_PROSE", "No manuscript prose found to scan. Draft first.")
        return

    if not os.path.exists(reference_text_path):
        log_status(CLR_RED, "FILE_NOT_FOUND", f"The reference file path '{reference_text_path}' does not exist.")
        return

    log_status(CLR_CYAN, "STAGE_4B", "Running Originality Scanner (originality-and-plagiarism-checker)...")

    try:
        with open(reference_text_path, "r", encoding="utf-8") as f:
            ref_material = f.read()
    except IOError as e:
        log_status(CLR_RED, "READ_ERR", f"Failed reading source file: {e}")
        return

    system_prompt = (
        "You are an IP compliance auditor and originality specialist for digital publications.\n"
        "Task: Scan the provided manuscript chapter against the original source materials to check for accidental verbatim copying or heavy structural mimicry."
    )
    user_prompt = (
        "Perform a thorough comparison check between the manuscript draft and the reference intake material.\n"
        "Report all findings using the format below:\n\n"
        "Originality Score: [Percentage 0-100%]\n"
        "Flagged Passages: [Quote manuscript vs source passages matching too closely, or 'None']\n"
        "Re-writing Directives: [If closeness exists, provide non-infringing rewrite directions, otherwise 'None']\n\n"
        f"=== INGESTED REFERENCE MATERIAL ===\n{ref_material[:5000]}\n===================================\n\n"
        f"=== GENERATED MANUSCRIPT CHAPTER ===\n{polished_text[:5000]}\n====================================\n"
    )

    log_status(CLR_CYAN, "API_CALL", "Transmitting draft text and reference text to Auditor subagent...")
    originality_report = request_openrouter_api(api_key, system_prompt, user_prompt)

    # Print to console
    print(f"\n{CLR_BOLD}{CLR_GREEN}=== IP COMPLIANCE & ORIGINALITY REPORT ==={CLR_RESET}\n")
    print(originality_report)

def export_manuscript() -> None:
    """Combines all polished chapters into a unified publish-ready manuscript."""
    project = load_project_state()
    title = project["project_metadata"].get("title", "Untitled Manuscript")

    log_status(CLR_CYAN, "EXPORT", f"Compiling complete manuscript chapters for '{title}'...")

    compiled_text = f"# {title.upper()}\n\n"
    compiled_text += "Generated via FinesseStories Narrative CLI Platform\n"
    compiled_text += "==================================================\n\n"

    found_chapters = 0
    sorted_keys = sorted(project["chapters"].keys(), key=lambda x: int(x.split("_")[1]) if "_" in x else 0)

    for key in sorted_keys:
        ch_num = key.split("_")[1]
        ch_data = project["chapters"][key]

        # Prefer polished prose, fallback to draft
        prose = ch_data.get("polished", "").strip()
        if not prose:
            prose = ch_data.get("draft", "").strip()

        if prose:
            found_chapters += 1
            compiled_text += f"\n\n## Chapter {ch_num}: {ch_data.get('title', 'Untitled Chapter')}\n"
            compiled_text += f"POV: {ch_data.get('pov_character', 'Protagonist')}\n"
            compiled_text += "--------------------------------------------------\n\n"
            compiled_text += prose + "\n"

    if found_chapters == 0:
        log_status(CLR_YELLOW, "WARNING", "No manuscript drafts found to export. Complete Stage 3 and 4 steps first.")
        return

    output_path = "compiled_manuscript.txt"
    try:
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(compiled_text)
        log_status(CLR_GREEN, "SUCCESS", f"Manuscript cleanly compiled! File exported to '{output_path}' ({found_chapters} chapters)")
    except Exception as e:
        log_status(CLR_RED, "WRITE_ERR", f"Could not export manuscript: {e}")

def run_interactive_shell(api_key: str) -> None:
    """Spins up a beautiful command-line shell with integrated slash commands."""
    print(f"\n{CLR_BOLD}{CLR_CYAN}==================================================================={CLR_RESET}")
    print(f"  {CLR_BOLD}{CLR_GREEN}FinesseStories CLI - High-Continuity Book Writing Workspace{CLR_RESET}")
    print(f"  Model Router: {CLR_YELLOW}OpenRouter Free Tier (Llama 3 / Qwen Coder / Gemma){CLR_RESET}")
    print(f"  Interactive Session Active. Type {CLR_BOLD}/help{CLR_RESET} to view commands.")
    print(f"{CLR_BOLD}{CLR_CYAN}==================================================================={CLR_RESET}\n")

    while True:
        try:
            user_input = input(f"{CLR_BOLD}{CLR_GREEN}finessestories>{CLR_RESET} ").strip()
            if not user_input:
                continue

            if user_input.startswith("/"):
                parts = user_input.split(maxsplit=2)
                cmd = parts[0].lower()
                arg1 = parts[1].strip() if len(parts) > 1 else ""
                arg2 = parts[2].strip() if len(parts) > 2 else ""

                if cmd in ["/exit", "/quit"]:
                    log_status(CLR_GREEN, "EXIT", "Closing FinesseStories CLI session. Happy writing!")
                    break

                elif cmd == "/help":
                    print(f"\n{CLR_BOLD}=== FinesseStories Workspace Commands ==={CLR_RESET}")
                    print("  /help                   Show this instructions helper menu")
                    print("  /status                 Show active Story Bible status & progress markers")
                    print("  /ingest <filepath>      Stage 1: Mine and extract raw premise & tropes")
                    print("  /world                  Stage 2a: Run Setting & Magic/Tech World-Building Architect")
                    print("  /characters             Stage 2b: Formulate deep Psychological Character Rosters")
                    print("  /bible                  Stage 2c: Compile Master Story Bible JSON (SSOT) from dossiers")
                    print("  /outline                Stage 2d: Formulate the detailed Chapter-by-Chapter Matrix")
                    print("  /draft <chapter>        Stage 3a: Draft narrative chapter with sliding context")
                    print("  /audit <chapter>        Stage 3b: Run Continuity Audit, enforce constraints")
                    print("  /polish <chapter>       Stage 4a: Polishing Pass, remove bad style and AI slop")
                    print("  /originality <ch> <fp>  Stage 4b: Check prose against original reference files")
                    print("  /export                 Compile and export the complete book manuscript file")
                    print("  /exit                   Safely close interactive session")
                    print()

                elif cmd == "/status":
                    project = load_project_state()
                    bible = load_story_bible()
                    meta = project.get("project_metadata", {})
                    brief = project.get("project_brief", {})
                    print(f"\n{CLR_BOLD}=== STORY WORKSPACE STATUS ==={CLR_RESET}")
                    print(f"  Working Title:       {CLR_BOLD}{meta.get('title')}{CLR_RESET}")
                    print(f"  Tone & Style:        {meta.get('tone_and_style')}")
                    print(f"  Intake Active:       {'YES (Brief Extracted)' if brief.get('core_premise') else 'NO'}")
                    print(f"  World Dossier:       {'YES (Setting Built)' if brief.get('world_dossier') else 'NO'}")
                    print(f"  Character Roster:    {'YES (Profiles Built)' if brief.get('character_roster') else 'NO'}")
                    print(f"  Story Bible Version: {bible.get('story_bible_version', 'Not compiled yet')}")
                    print(f"  Factions Count:      {len(bible.get('world', {}).get('factions', []))}")
                    print(f"  Characters Listed:   {len(bible.get('characters', []))}")
                    print(f"  Chapter Outlines:    {len(project.get('chapters', {}).keys())} chapter(s) registered")
                    print(f"  Canon Updates Ledger: {len(bible.get('chronology_and_canon_ledger', []))} audited facts logged")
                    print()

                elif cmd == "/ingest":
                    if not arg1:
                        log_status(CLR_RED, "ERR", "Specify target reference file. Usage: /ingest <filepath>")
                    else:
                        run_stage1_ingest(api_key, arg1)

                elif cmd == "/world":
                    run_stage2_world(api_key)

                elif cmd == "/characters":
                    run_stage2_characters(api_key)

                elif cmd == "/bible":
                    run_stage2_architect(api_key)

                elif cmd == "/outline":
                    run_stage2_outliner(api_key)

                elif cmd == "/draft":
                    if not arg1:
                        log_status(CLR_RED, "ERR", "Specify chapter number. Usage: /draft <chapter_num>")
                    else:
                        try:
                            ch = int(arg1)
                            run_stage3_draft(api_key, ch)
                        except ValueError:
                            log_status(CLR_RED, "ERR", "Chapter must be a valid integer.")

                elif cmd == "/audit":
                    if not arg1:
                        log_status(CLR_RED, "ERR", "Specify chapter number. Usage: /audit <chapter_num>")
                    else:
                        try:
                            ch = int(arg1)
                            run_stage3_auditor(api_key, ch)
                        except ValueError:
                            log_status(CLR_RED, "ERR", "Chapter must be a valid integer.")

                elif cmd == "/polish":
                    if not arg1:
                        log_status(CLR_RED, "ERR", "Specify chapter number. Usage: /polish <chapter_num>")
                    else:
                        try:
                            ch = int(arg1)
                            run_stage4_polish(api_key, ch)
                        except ValueError:
                            log_status(CLR_RED, "ERR", "Chapter must be a valid integer.")

                elif cmd == "/originality":
                    if not arg1 or not arg2:
                        log_status(CLR_RED, "ERR", "Usage: /originality <chapter_num> <reference_file_path>")
                    else:
                        try:
                            ch = int(arg1)
                            run_stage4_originality(api_key, ch, arg2)
                        except ValueError:
                            log_status(CLR_RED, "ERR", "Chapter must be a valid integer.")

                elif cmd == "/export":
                    export_manuscript()

                else:
                    log_status(CLR_RED, "ERR", f"Unknown slash command '{cmd}'. Type /help for assistance.")
            else:
                log_status(CLR_YELLOW, "REPL", "Raw inputs must be sent via slash commands. Type /help to see options.")

        except (KeyboardInterrupt, EOFError):
            print()
            log_status(CLR_GREEN, "SHELL", "Session closed dynamically. Goodbye!")
            break

def main() -> None:
    parser = argparse.ArgumentParser(
        description="FinesseStories Narrative Production Suite CLI",
        formatter_class=argparse.RawDescriptionHelpFormatter
    )
    subparsers = parser.add_subparsers(dest="command")

    # Ingest Command Setup
    ingest_parser = subparsers.add_parser("ingest", help="Stage 1: Mine and extract core parameters from source file")
    ingest_parser.add_argument("reference", type=str, help="Text reference or transcript file path")
    ingest_parser.add_argument("--key", type=str, help="OpenRouter API key")

    # World Architect Setup
    subparsers.add_parser("world", help="Stage 2a: Generate Setting and Systems Dossier")

    # Character Architect Setup
    subparsers.add_parser("characters", help="Stage 2b: Generate Character Master Roster Profiles")

    # Bible Compiler Setup
    subparsers.add_parser("bible", help="Stage 2c: Generate unified Story Bible (SSOT) JSON from Dossiers")

    # Outliner Setup
    subparsers.add_parser("outline", help="Stage 2d: Create Chapter Outline Matrix from Story Bible")

    # Draft Chapter Setup
    draft_parser = subparsers.add_parser("draft", help="Stage 3a: Write Chapter raw prose using sliding summaries")
    draft_parser.add_argument("chapter", type=int, help="Target Chapter number")
    draft_parser.add_argument("--key", type=str, help="OpenRouter API key")

    # Auditor Setup
    audit_parser = subparsers.add_parser("audit", help="Stage 3b: Evaluate prose against Story Bible canon constraints")
    audit_parser.add_argument("chapter", type=int, help="Target Chapter number")
    audit_parser.add_argument("--key", type=str, help="OpenRouter API key")

    # Polish Setup
    polish_parser = subparsers.add_parser("polish", help="Stage 4a: Line polishing pass to remove AI slop")
    polish_parser.add_argument("chapter", type=int, help="Target Chapter number")
    polish_parser.add_argument("--key", type=str, help="OpenRouter API key")

    # Plagiarism/Originality Setup
    originality_parser = subparsers.add_parser("originality", help="Stage 4b: Verify uniqueness against reference files")
    originality_parser.add_argument("chapter", type=int, help="Target Chapter number")
    originality_parser.add_argument("reference", type=str, help="Original Source reference document path")
    originality_parser.add_argument("--key", type=str, help="OpenRouter API key")

    # Export Setup
    subparsers.add_parser("export", help="Compile and format all completed manuscript chapters")

    # Shell Setup
    shell_parser = subparsers.add_parser("shell", help="Launch interactive workspace shell with slash commands")
    shell_parser.add_argument("--key", type=str, help="OpenRouter API key")

    args = parser.parse_args()
    api_key = getattr(args, "key", None) or os.environ.get("OPENROUTER_API_KEY", "")

    if args.command == "ingest":
        run_stage1_ingest(api_key, args.reference)
    elif args.command == "world":
        run_stage2_world(api_key)
    elif args.command == "characters":
        run_stage2_characters(api_key)
    elif args.command == "bible":
        run_stage2_architect(api_key)
    elif args.command == "outline":
        run_stage2_outliner(api_key)
    elif args.command == "draft":
        run_stage3_draft(api_key, args.chapter)
    elif args.command == "audit":
        run_stage3_auditor(api_key, args.chapter)
    elif args.command == "polish":
        run_stage4_polish(api_key, args.chapter)
    elif args.command == "originality":
        run_stage4_originality(api_key, args.chapter, args.reference)
    elif args.command == "export":
        export_manuscript()
    elif args.command == "shell":
        run_interactive_shell(api_key)
    else:
        # Default behavior is to enter the interactive shell mode
        run_interactive_shell(api_key)

if __name__ == "__main__":
    main()

usage: colab_kernel_launcher.py [-h]
                                {ingest,world,characters,bible,outline,draft,audit,polish,originality,export,shell}
                                ...
colab_kernel_launcher.py: error: argument command: invalid choice: '/root/.local/share/jupyter/runtime/kernel-ff64db91-23b2-4155-996c-935372c62326.json' (choose from ingest, world, characters, bible, outline, draft, audit, polish, originality, export, shell)
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/lib/python3.12/argparse.py", line 1943, in _parse_known_args2
    namespace, args = self._parse_known_args(args, namespace, intermixed)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 2188, in _parse_known_args
    stop_index = consume_positionals(start_index)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 2141, in consume_positionals
    take_action(action, args)
  File "/usr/lib/python3.12/argparse.py", line 2003, in take_action
    argument_values = self._get_values(action, argument_strings)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 2523, in _get_values
    self._check_value(action, value[0])
  File "/usr/lib/python3.12/argparse.py", line 2573, in _check_value
    raise ArgumentError(action, msg % args)
argparse.ArgumentError: argument command: i

TypeError: object of type 'NoneType' has no len()